# Stage-2 crop segmentation — unified backbone training

Trains the second-stage crop segmenter (SMP U-Net) that turns each detected line box
into a polygon mask, for `(BACKBONE, FAMILY, SUBSET)`. Reuses `01_simple_segmentation`'s
`SMPUNet` so the **same 9 backbones** (incl. hier_encoder SFP for ViT/DINO/RADIO) are
available here as for detection. Replaces `crops_unet_vs_yolo`.

## Parameters

In [ ]:
# ==================== PARAMETERS (identical block in every 02_2stage notebook) ====================
# Switch the whole notebook by editing these three. BACKBONE is any key from
# train_detector.BACKBONES; FAMILY/SUBSET pick the dataset subset.
BACKBONE = "dinov2"      # resnet34 resnet50 convnext convnext_dinov3
                          # vit_small_patch16_224.augreg_in21k vit_small_patch16_dinov3
                          # dinov2 dinov2_reg am-radio
FAMILY   = "diva"        # "diva" | "udiads"
SUBSET   = "CB55"        # diva: CB55 CS18 CS863   |   udiads: Latin14396 Latin2 Syr341
IMGSZ    = 1280           # detector inference/training size (multiple of 32)
CONF     = 0.25           # detector confidence threshold
CROP_IMGSZ = 640          # stage-2 crop-segmenter size

## Resolve paths + map BACKBONE → SMP encoder name

In [ ]:
import sys, torch
from pathlib import Path
sys.path.insert(0, ".")
import train_detector

assert BACKBONE in train_detector.BACKBONES, f"unknown BACKBONE {BACKBONE!r}"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# (FAMILY, SUBSET) -> detection data config + trained-detector checkpoint path.
DATA_CFG = Path(f"configs/data/{('diva_'+SUBSET.lower()) if FAMILY=='diva' else ('udiads_'+SUBSET.lower())}_detect.yaml")
FAM_DIR = "diva-hisdb" if FAMILY == "diva" else "u-diads-tl"
RUN = f"yolov8_{BACKBONE}_{FAMILY}_{SUBSET}"
DET_WEIGHTS = Path(f"../../80_models/02_2stage/{FAM_DIR}/detection/{RUN}/weights/best.pt")
print("device      :", DEVICE)
print("data config :", DATA_CFG, "(exists)" if DATA_CFG.exists() else "(MISSING)")
print("detector    :", DET_WEIGHTS, "(exists)" if DET_WEIGHTS.exists() else "(train it first)")

# The SMP crop-segmenter uses 01_simple_segmentation's encoder names; map the shared
# BACKBONE key to the SMP encoder (ConvNeXt goes through timm's "tu-" prefix there).
SMP_ENCODER = {
    "resnet34": "resnet34",
    "resnet50": "resnet50",
    "convnext": "tu-convnext_tiny.in12k_ft_in1k",
    "convnext_dinov3": "tu-convnext_tiny.dinov3_lvd1689m",
    "vit_small_patch16_224.augreg_in21k": "vit_small_patch16_224.augreg_in21k",
    "vit_small_patch16_dinov3": "vit_small_patch16_dinov3",
    "dinov2": "dinov2",
    "dinov2_reg": "dinov2_reg",
    "am-radio": "am-radio",
}[BACKBONE]
print("SMP encoder :", SMP_ENCODER)

# Reuse 01's SMPUNet (+ vit_hier SFP registration) for backbone parity.
sys.path.insert(0, "../01_simple_segmentation")
from models.smp_unet import SMPUNet
# Absolute data YAML (fixes Ultralytics resolving `path:` vs datasets_dir).
DATA = train_detector.resolve_data(DATA_CFG) if DATA_CFG.exists() else str(DATA_CFG)


## Build crop dataset (GT polygon → bbox crop + polygon mask)

In [ ]:
import yaml as _yaml
from dataset import build_segmentation_dataset
# The crop dataset is built from the same detection data config's split layout.
cfg = _yaml.safe_load(open(DATA))
cfg["encoder_name"] = SMP_ENCODER
cfg["crop_imgsz"] = CROP_IMGSZ
train_ds = build_segmentation_dataset(cfg, split="train")
val_ds   = build_segmentation_dataset(cfg, split="val")
print("train crops:", len(train_ds), " val crops:", len(val_ds))

## Train

In [ ]:
import torch
from torch.utils.data import DataLoader
model = SMPUNet({"model": {"arch": "unet", "encoder_name": SMP_ENCODER}}).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = torch.nn.BCEWithLogitsLoss()
EPOCHS = 50
dl = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
for ep in range(EPOCHS):
    model.train(); tot = 0.0
    for img, mask in dl:
        img, mask = img.to(DEVICE), mask.to(DEVICE)
        opt.zero_grad(); out = model(img); loss = loss_fn(out, mask)
        loss.backward(); opt.step(); tot += loss.item()
    print(f"epoch {ep+1}/{EPOCHS}  loss {tot/len(dl):.4f}")

## Save checkpoint

In [ ]:
OUT = Path(f"../../80_models/02_2stage/{FAM_DIR}/crop_seg/seg_{BACKBONE}_{FAMILY}_{SUBSET}")
OUT.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), OUT / "best.pth")
print("saved ->", OUT / "best.pth")